# Algorithms

## Libraries

In [3]:
import numpy as np
from pprint import pprint
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Set
import pydot
from IPython.display import Image
from collections import deque
from typing import List, Any, Tuple
import copy
import random
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from pprint import pprint
import networkx as nx
import matplotlib.pyplot as plt
import pydot
from IPython.display import Image
from collections import deque
from typing import List, Any, Tuple
import networkx as nx
import matplotlib.pyplot as plt
import copy

In [4]:

def activation_derivative(output_value: float):
    return 1 - output_value ** 2

modelos_clasification = {
    'LogisticRegression': LogisticRegression(),
    'RandomForest': RandomForestClassifier(),
    'RidgeClassifier': RidgeClassifier(),
    'KNN': KNeighborsClassifier(),
    'Xg': XGBClassifier(),
    'SVC': SVC()
}

meta_param_grid= {
    'KNN': {'n_neighbors': [i for i in range(1, 20)]},
    'RidgeClassifier': {'alpha': [0.1, 1, 10, 100]},
    'LogisticRegression': {'C': [0.1, 1, 10, 100], 'penalty': ['l1', 'l2']},
    'RandomForest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20, 30], 'criterion': ['gini', 'entropy']},
    'Xg': {'max_depth': [3, 4, 5, 6, 7], 'learning_rate': [0.1, 0.2, 0.3], 'n_estimators': [100, 200, 300]},
    'SVC': {'kernel': ['linear', 'poly', 'rbf', 'sigmoid'], 'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto'], 'degree': [2, 3, 4]}
}
class Gen_neuron:
    def __init__(self, neuron_id: int, neuron_type: str, bias: float) -> None:
        self.neuron_id = neuron_id
        self.neuron_type = neuron_type
        self.output_value = 0.0
        self.bias = bias
    def __repr__(self):
        return (f"Gen_neuron("
                f"id={self.neuron_id}, "
                f"type='{self.neuron_type}', "
                f"bias={self.bias}, "
                f"output_value={self.output_value})")
    
class Neuron:
    def __init__(self, gen_neuron: Gen_neuron) -> None:
        self.neuron_id = gen_neuron.neuron_id
        self.neuron_type = gen_neuron.neuron_type
        self.output_value = gen_neuron.output_value
        self.bias = gen_neuron.bias
        self.activation_function = np.tanh
        self.activation_derivative = activation_derivative
        self.delta = 0.0  
    def __repr__(self):
        return (f"Neuron("
                f"id={self.neuron_id}, "
                f"type='{self.neuron_type}', "
                f"bias={self.bias}, "
                f"output_value={self.output_value}, "
                f"delta={self.delta})")

class Gen_connection:
    def __init__(self, from_neuron_id: int, to_neuron_id: int, weight: float, enabled: bool = True) -> None:
        self.from_neuron_id = from_neuron_id
        self.to_neuron_id = to_neuron_id
        self.weight = weight
        self.enabled = enabled
    def __repr__(self):
        return (f"Gen_connection("
                f"from_neuron_id={self.from_neuron_id}, "
                f"to_neuron_id={self.to_neuron_id}, "
                f"weight={self.weight}, "
                f"enabled={self.enabled})")

class Connection:
    def __init__(self, gen_connection: Gen_connection) -> None:
        self.from_neuron_id = gen_connection.from_neuron_id
        self.to_neuron_id   = gen_connection.to_neuron_id
        self.weight         = gen_connection.weight
        self.enabled        = gen_connection.enabled
    def __repr__(self):
        return (f"Connection("
                f"from_neuron_id={self.from_neuron_id}, "
                f"to_neuron_id={self.to_neuron_id}, "
                f"weight={self.weight}, "
                f"enabled={self.enabled})")
        
class Genotype:
    def __init__(self) -> None:
        self.neurons = []
        self.connections = []
        self.layers = []

    def add_neuron(self, neuron: Gen_neuron):
        self.neurons.append(neuron)

    def add_connection(self, connection: Gen_connection):
        self.connections.append(connection)

    def stablish_layers(self, layers_array):
        self.layers = layers_array

    def delete_connection(self):
        self.connections = self.connections[:-1]

    def mutate(self):
        def dfs_path_exists_genotype(genotype: Genotype, start_id: int, end_id: int) -> bool:
            visited: Set[int] = set()
            path_found = False  # Verifica si al menos un camino llega al nodo de destino

            def dfs(current_id: int) -> None:
                nonlocal path_found
                # Si llegamos al nodo objetivo, registramos el camino encontrado
                if current_id == end_id:
                    path_found = True
                    return

                # Marcamos la neurona actual como visitada
                visited.add(current_id)

                # Iterar sobre las conexiones salientes de la neurona actual
                for connection in genotype.connections:
                    if connection.enabled and connection.from_neuron_id == current_id and connection.to_neuron_id not in visited:
                        # Realizar la llamada recursiva para explorar el siguiente nodo
                        dfs(connection.to_neuron_id)

                # Removemos el nodo actual de visitados para permitir otros caminos
                visited.remove(current_id)

            # Llamar al DFS desde el nodo inicial
            dfs(start_id)
            return path_found
        def attempt_mutation(mutation_function):
            for _ in range(5):
                # Crear una copia del genotipo actual para probar la mutación
                test_genotype = copy.deepcopy(self)
                mutation_function(test_genotype)

                # Lista para registrar si existe un camino desde cada entrada a cada salida
                path_exists = []

                # Iterar sobre cada neurona de entrada y salida
                for input_neuron in test_genotype.layers[0]:  # Neuronas de entrada
                    for output_neuron in test_genotype.layers[-1]:  # Neuronas de salida
                        start_neuron_id = input_neuron.neuron_id
                        end_neuron_id = output_neuron.neuron_id
                        
                        # Realizar la búsqueda DFS para verificar si existe un camino desde la entrada a la salida
                        exists = dfs_path_exists_genotype(test_genotype, start_neuron_id, end_neuron_id)
                        print(f'DFS ({start_neuron_id} -> {end_neuron_id}): {exists}')
                        path_exists.append(exists)

                # Verificar si el grafo sigue siendo acíclico y si todas las entradas llegan a todas las salidas
                if kahn_topological_order(Network(test_genotype)) is not None and all(path_exists):
                    print('Mutación válida, se realizó un cambio')
                    print('Caminos validados:', path_exists)
                    print('Función de mutación aplicada:', str(mutation_function))
                    # Aplicar la mutación al genotipo original si la prueba fue exitosa
                    mutation_function(self)
                    return
                elif kahn_topological_order(Network(test_genotype)) is None:
                    print('Grafo cíclico detectado, mutación rechazada')
                elif not all(path_exists):
                    print('No todos los caminos existen después de la mutación:', path_exists)
        def add_random_connection(test_genotype):
            """Agrega una conexión aleatoria entre dos neuronas no conectadas."""
            possible_connections = []
            for n1 in test_genotype.neurons:
                for n2 in test_genotype.neurons:
                    if n1.neuron_id != n2.neuron_id:
                        connection_exists = any(
                            conn.from_neuron_id == n1.neuron_id and conn.to_neuron_id == n2.neuron_id 
                            for conn in test_genotype.connections
                        )
                        inverse_connection_exists = any(
                            conn.from_neuron_id == n2.neuron_id and conn.to_neuron_id == n1.neuron_id 
                            for conn in test_genotype.connections
                        )
                        if not connection_exists and not inverse_connection_exists and n1.neuron_type != 'output' and n2.neuron_type != 'input':
                            possible_connections.append((n1.neuron_id, n2.neuron_id))
                            
            if possible_connections:
                from_id, to_id = random.choice(possible_connections)
                new_weight = random.uniform(-1.0, 1.0)
                new_connection = Gen_connection(from_id, to_id, new_weight)
                test_genotype.add_connection(new_connection)

        def delete_random_connection(test_genotype):
            """Elimina una conexión aleatoria del genotipo."""

            if test_genotype.connections:
                connection_to_delete = random.choice(test_genotype.connections)
                test_genotype.connections.remove(connection_to_delete)

        def modify_random_weight(test_genotype):
            """Modifica aleatoriamente el peso de una conexión."""
            if test_genotype.connections:
                connection = random.choice(test_genotype.connections)
                connection.weight += random.uniform(-0.5, 0.5)

        def add_random_neuron(test_genotype):
            """Agrega una nueva neurona entre dos conexiones existentes."""
            if test_genotype.connections:
                connection = random.choice(test_genotype.connections)
                if connection.enabled:
                    # Crear nueva neurona
                    new_neuron_id = len(test_genotype.neurons)
                    new_bias = random.uniform(-0.5, 0.5)
                    new_neuron = Gen_neuron(new_neuron_id, "hidden", new_bias)
                    test_genotype.add_neuron(new_neuron)
                    connection.enabled = False  # Temporarily disable the connection
                    test_genotype.add_connection(Gen_connection(connection.from_neuron_id, new_neuron_id, random.uniform(-1.0, 1.0)))
                    test_genotype.add_connection(Gen_connection(new_neuron_id, connection.to_neuron_id, random.uniform(-1.0, 1.0)))

                    # Update the layers to include the new neuron in the appropriate position
                    for layer in test_genotype.layers:
                        if connection.from_neuron_id in [neuron.neuron_id for neuron in layer]:
                            layer.append(new_neuron)
                            break

        def modify_random_bias(test_genotype):
            """Modifica el sesgo de una neurona aleatoria."""
            if test_genotype.neurons:
                neuron = random.choice(test_genotype.neurons)
                
                neuron.bias += random.uniform(-0.1, 0.1)

        # Lista de funciones de mutación
        mutation_functions = [
            add_random_connection,
            delete_random_connection,
            modify_random_weight,
            add_random_neuron,
            modify_random_bias,
        ]

        mutation_function = random.choice(mutation_functions)
        attempt_mutation(mutation_function)

class Network:
    def __init__(self, genotype: Genotype) -> None:
        self.neurons = [Neuron(neuron) for neuron in genotype.neurons]
        self.connections = [Connection(connection) for connection in genotype.connections]
        self.layers = [[self.neurons[genotype.neurons.index(neuron)] for neuron in layer] for layer in genotype.layers]
    def __repr__(self):
        neurons_repr = ", ".join(repr(neuron) for neuron in self.neurons)
        connections_repr = ", ".join(repr(connection) for connection in self.connections)
        layers_repr = ", ".join(f"[{', '.join(repr(neuron) for neuron in layer)}]" for layer in self.layers)
        return (f"Network(\n"
                f"  Neurons=[{neurons_repr}],\n"
                f"  Connections=[{connections_repr}],\n"
                f"  Layers=[{layers_repr}]\n"
                f")")
    
    def forward_propagator(self, input_values: List[Any]):
        order = kahn_topological_order(self)
        for index, (neuron, input_value) in enumerate(zip(self.layers[0], input_values)):
            if neuron.neuron_type == 'input':
                neuron.output_value = input_value
                self.neurons[index] = neuron
        for neuron_id in order:
            neuron = self.neurons[neuron_id]
            if neuron.neuron_type != 'input':
                weighted_inputs = []
                for connection in self.connections:
                    if connection.to_neuron_id == neuron.neuron_id and connection.enabled:
                        from_neuron = self.neurons[connection.from_neuron_id]
                        weighted_inputs.append(from_neuron.output_value * connection.weight)
                neuron.output_value = neuron.activation_function(sum(weighted_inputs) - neuron.bias)
                self.neurons[neuron_id] = neuron

        return [neuron.output_value for neuron in self.layers[-1]]
    def backward_propagator(self, target_values: List[float], learning_rate: float = 0.01):
        '''
        Implementa el algoritmo de retropropagación para ajustar los pesos de la red.
        '''
        errors = []
        order = kahn_topological_order(self)
        reversed_order = order[::-1]
        for i, output_neuron in enumerate(self.layers[-1]):
            output_error = target_values[i] - output_neuron.output_value
            errors.append(output_error)
            output_neuron.delta = output_error * output_neuron.activation_derivative(output_neuron.output_value)
        for neuron_id in reversed_order:
            neuron = self.neurons[neuron_id]
            if neuron.neuron_type != 'output':
                error_sum = 0.0
                for connection in self.connections:
                    if connection.from_neuron_id == neuron.neuron_id and connection.enabled:
                        to_neuron = self.neurons[connection.to_neuron_id]
                        error_sum += to_neuron.delta * connection.weight
                neuron.delta = error_sum * neuron.activation_derivative(neuron.output_value)
        for connection in self.connections:
            if connection.enabled:
                from_neuron = self.neurons[connection.from_neuron_id]
                to_neuron = self.neurons[connection.to_neuron_id]
                connection.weight += learning_rate * to_neuron.delta * from_neuron.output_value
                from_neuron.bias += learning_rate * to_neuron.delta

def kahn_topological_order(network: Network):
    neurons = network.neurons
    if network.connections:
        connections = network.connections
        in_degree = {neuron.neuron_id: 0 for neuron in neurons}
        adj_list = {neuron.neuron_id: [] for neuron in neurons}
        for connection in connections:
            if connection.enabled:
                adj_list[connection.from_neuron_id].append(connection.to_neuron_id)
                in_degree[connection.to_neuron_id] += 1
        queue = deque([neuron_id for neuron_id in in_degree if in_degree[neuron_id] == 0])
        topological_order = []
        while queue:
            current = queue.popleft()
            topological_order.append(current)
            for neighbor in adj_list[current]:
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)
        if len(topological_order) != len(neurons):
            return None

        return topological_order
    else:
        return []
class Population():
    def __init__(self, population_id, number_gens):
        self.population_id = population_id
        self.number_gens   = number_gens
        self.populationGenes = None
        self.populationNets  = None 
    def initialize_population(self):
        populationGenes = []
        populationNets  = []
        for _ in range(self.number_gens):
            Num_neurons_input  = 1
            Num_neurons_hidden = np.random.randint(3,7)
            Num_neurons_output = 2
            Gen = Genotype()
            G   = nx.DiGraph()
            Node_generator(Gen, Num_neurons_hidden, Num_neurons_input, Num_neurons_output, [])
            player_X_caller(Gen)
            Net           = Network(Gen)
            populationGenes.append(Gen)
            populationNets.append(Net)
        self.populationGenes = populationGenes
        self.populationNets  = populationNets

        return populationGenes, populationNets
    def __to_data_test_train(self, index_population, df, populationNets, orders_list):
        new_rows = []
        for net in populationNets:
            num_hidden = len(net.layers[1])
            num_connections = len(net.connections)
            average_weight = np.mean([conn.weight for conn in net.connections])
            average_bias = np.mean([neuron.bias for neuron in net.neurons])
            new_row = [num_hidden, num_connections, average_weight, average_bias, 0, 0]
            new_rows.append(new_row)
        new_df_rows = pd.DataFrame(new_rows, columns=["num_hidden", "num_connections", "average_weight", "average_bias", "order", "target"])
        new_df = pd.concat([df, new_df_rows], axis=0, ignore_index=True)
        print(len(new_df),len(orders_list))
        new_df['order'] = orders_list
        new_df.to_excel(f'network_dataframes/{index_population}_NetworkData.xlsx',index=False)
        return new_df    
    
    def to_dataframe(self, index_population, df, populationNets, orders_list):
        
        return self.__to_data_test_train(index_population, df, populationNets, orders_list)
    def forward_population(self, input_values):
        orders = []
        for net in self.populationNets:
            input_value =  [np.random.choice(input_values)]
            order = net.forward_propagator(input_value)
            orders.append(order.index(max(order)))
        return orders
    def Natural_Selection(self, df):
        X_train = (df.drop(['target'],axis='columns'))[:-200]
        y_train = (df.target)[:-200]
        X_test  = (df.drop('target',axis='columns'))[-200:]
        y_test  = (df.target)[-200:]
        
        results = {}
        for model_name, model in modelos_clasification.items():
            print(f"Training model: {model_name}")
            if model_name in meta_param_grid:
                param_grid = meta_param_grid[model_name]
                grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=5, n_jobs=-1)
                grid_search.fit(X_train, y_train)
                
                best_model = grid_search.best_estimator_
                y_pred = best_model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                
                results[model_name] = {
                    'Best Model': best_model,
                    'Best Parameters': grid_search.best_params_,
                    'Accuracy': accuracy
                }
        return results

        



         
def plot_network(genotype: Genotype):
    pos = {}
    def find_position(neuron,positions):
            if neuron.neuron_type=='input':
                while True:
                    position = (-7, np.random.randint(-10, 10))
                    if position not in positions:
                        pos[neuron.neuron_id] = position
                        positions.append(position)
                        break
            if neuron.neuron_type=='hidden':
                while True:
                    position = (np.random.randint(-4, 4), np.random.randint(-10, 10))
                    if position not in positions:
                        pos[neuron.neuron_id] = position
                        positions.append(position)
                        break                     
            elif neuron.neuron_type=='output':
                while True:
                    position = (7, np.random.randint(-10, 10))
                    if position not in positions:
                        pos[neuron.neuron_id] = position
                        positions.append(position)
                        break 
    G = nx.DiGraph()
    for neuron in genotype.neurons:
        G.add_node(neuron.neuron_id, type=neuron.neuron_type)
    for connection in genotype.connections:
        if connection.enabled:
            G.add_edge(connection.from_neuron_id, connection.to_neuron_id)
    
    positions = []
    for i, layer in enumerate(genotype.layers):
        for j, neuron in enumerate(layer):
            find_position(neuron, positions)
    plt.figure(figsize=(10, 6))
    nx.draw(G, pos, with_labels=True, node_size=700, node_color='#ff721c', font_size=10, font_weight='bold', arrows=True)
    labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)
    plt.title("Neural Network Structure")
    plt.show()

def Net_Gen_road_creator( Gen: Genotype, tuple_nodes: Tuple, list_connections: List[List[int]], dx):
    if tuple_nodes not in list_connections and tuple_nodes[::-1] not in list_connections and dx[tuple_nodes[1]] < 3 and tuple_nodes[0]!=tuple_nodes[1]:
        weight = np.random.uniform(0, 1)
        conn = Gen_connection(tuple_nodes[0], tuple_nodes[1], weight)
        Net_copy = copy.deepcopy(Gen)
        Net_copy.add_connection(conn)
        order = kahn_topological_order(Network(Net_copy))
        if order is None:
            return None
        Gen.add_connection(conn)
        dx[tuple_nodes[1]] += 1
        list_connections.append(tuple_nodes)

def Node_generator(Gen: Genotype, Num_neurons_hidden, Num_neurons_input, Num_neurons_output, sub):
    positions = []
    for ID in range(Num_neurons_hidden + Num_neurons_input + Num_neurons_output):
        if ID < Num_neurons_input:
            bias = np.random.uniform(0, 0.3)
            neuron = Gen_neuron(neuron_id=ID, neuron_type='input', bias=bias)
            sub.append(neuron)
            Gen.add_neuron(neuron)
        elif ID >= Num_neurons_input and ID < Num_neurons_hidden + Num_neurons_input:
            bias = np.random.uniform(0, 0.3)
            neuron = Gen_neuron(neuron_id=ID, neuron_type='hidden', bias=bias)
            sub.append(neuron)
            Gen.add_neuron(neuron)
        else:
            bias = np.random.uniform(0, 0.3)
            neuron = Gen_neuron(neuron_id=ID, neuron_type='output', bias=bias)
            sub.append(neuron)
            Gen.add_neuron(neuron)

    Gen.layers.append(sub[:Num_neurons_input])
    Gen.layers.append(sub[Num_neurons_input:Num_neurons_hidden + Num_neurons_input])
    Gen.layers.append(sub[Num_neurons_hidden + Num_neurons_input:])

def player_X_caller(Gen: Genotype, dx=None) -> None:
    if dx is None:
        dx = {i: 0 for i in range(len(Gen.neurons))}
    set_inputs, set_outputs, set_hidden = set(), set(), set()
    list_connections = []
    while True:
        current_neuron = Gen.neurons.index(np.random.choice(Gen.layers[-1]))
        set_outputs.add(current_neuron)
        num_pre_neurons = np.random.randint(1, len(Gen.layers[1]))
        pre_neurons = np.random.choice(Gen.layers[1], size=num_pre_neurons, replace=False)
        for pre_neuron in pre_neurons:
            if pre_neuron != current_neuron:
                tuple_nodes = [Gen.neurons.index(pre_neuron), current_neuron]
                Net_Gen_road_creator(Gen, tuple_nodes, list_connections, dx)
                if tuple_nodes in list_connections:
                    current_neuron = tuple_nodes[0]
                    set_hidden.add(current_neuron)
        last_neuron = Gen.neurons.index(np.random.choice(Gen.layers[0]))
        tuple_nodes = [last_neuron, current_neuron]
        Net_Gen_road_creator(Gen, tuple_nodes, list_connections, dx)
        if tuple_nodes in list_connections:
            current_neuron = tuple_nodes[0]
            set_inputs.add(current_neuron)
        if (len(set_outputs) == len(Gen.layers[-1]) and len(set_inputs) == len(Gen.layers[0]) and len(set_hidden) == len(Gen.layers[1])):
            break
    


In [ ]:
# df_prices = pd.read_csv('prices_for_models.txt',delimiter=',')
# df_prices['date'] = pd.to_datetime(df_prices['date'])
# df_prices['time'] = pd.to_datetime(df_prices['time'], format='%H:%M').dt.time

In [ ]:
index_population = 0
len_population = 1000
pop = Population(index_population, len_population)
#trade_station = Trade_Station(financial_instrument="S&P 500")
gens, nets = pop.initialize_population()
df = pd.DataFrame(columns=["num_hidden", "num_connections", "average_weight", "average_bias", "order", "target"])
# last_date = df_prices['date'].iloc[0]
current_hour = None
price_list = []
order_dict = {
    "long": {"open": "Buy", "close": "Sell"},
    "short": {"open": "Sell short", "close": "Buy to cover"}
}
orders_list = []
id = 0
Generation = 0
date = 1
# for row in range(len(df_prices)):
#     # date = (df_prices.iloc[row])['date']
#     # time = (df_prices.iloc[row])['time']
#     # bid  = (df_prices.iloc[row])['bid']
#     # ask  = (df_prices.iloc[row])['ask']
#     if (date - last_date).days >= 1:
        # print('begining of selection...')
        # ruta_carpeta = "network_dataframes"

        # archivos = os.listdir(ruta_carpeta)
        # archivos_excel = [archivo for archivo in archivos if archivo.endswith(('.xlsx', '.xls'))]

        # if len(archivos_excel) >= 1:
        #     selected_archivos = random.sample(archivos_excel, 6)
        # else:
        #     selected_archivos = archivos_excel  
        # dataframes = []
        # for archivo in selected_archivos:
        #     ruta_archivo = os.path.join(ruta_carpeta, archivo)
        #     try:
        #         df = pd.read_excel(ruta_archivo)  
        #         dataframes.append(df)            
        #     except Exception as e:
        #         pass
        # if dataframes:
        #     df_final = pd.concat(dataframes, ignore_index=True)
    #     #     orders_list = []
    #     #     path_directory = f'results/Generation_{Generation}'
    #     #     os.makedirs(path_directory, exist_ok=True)
    #     #     pop.Natural_Selection(df_final, path_directory)
    #     #     pop.mutate()
    #     #     Generation += 1
    #     #     last_date         = date
    #     # for archivo in archivos_excel:
    #     #     ruta_archivo = os.path.join(ruta_carpeta, archivo)
    #     #     try:
    #     #         os.remove(ruta_archivo)  
    #     #     except Exception as e:
    #     #         pass


    # if  time.minute == 0:
    #         if current_hour is None and price_list:
    #             #df = pd.DataFrame(columns=["num_hidden", "num_connections", "average_weight", "average_bias", "order", "target"])
    #             #orders_list = pop.forward_population(price_list) 
    #             #df = pop.to_dataframe(index_population, df, nets, orders_list)
    #             # price_list  = []
    #             # current_hour = time.hour
    #             # if orders_list:
    #             #     chosen_order = mode(orders_list)[0]
    #             #     if chosen_order == 0:
    #             #             trade_station.Place_Order(id, False, date, time, bid, "Sell short")
    #             #     elif chosen_order == 1:
    #             #             trade_station.Place_Order(id, False, date, time, ask, "Buy")
    #             # long_price = trade_station.Place_Order(id, True, date, time, ask, order_dict["long"]["open"])
    #             # current_hour = time.hour
    #             # short_price = trade_station.Place_Order(id, True, date, time, bid, order_dict["short"]["open"])
    #             id += 1 
    # elif time.minute == 55:
    #             if current_hour is not None:
    #                 reality = 0
    #                 # if long_price > ask:
    #                 #        reality = 1  
    #                 # trade_station.Place_Order(id, True, date, time, bid, order_dict["long"]["close"])
    #                 # trade_station.Place_Order(id, True, date, time, ask, order_dict["short"]["close"])
    #                 # if chosen_order == 0:
    #                 #         trade_station.Place_Order(id, False, date, time, ask, "Buy to cover") 
    #                 # elif chosen_order == 1:
    #                 #         trade_station.Place_Order(id, False, date, time, bid, "Sell")
    #                 # id += 1
    #                 # price_list = []
    #                 # pop.stablish_usefullnes(index_population, df, reality)
    #                 index_population +=   1  
    #                 current_hour = None    
    # avg_price = (ask + bid) / 2
    # price_list.append(avg_price)